In [1670]:
# Variational autoencoder in Numpy without AI or reference code
#
# Honest list of all resources used
#   Feed-forward networks: Michael Nielsen, Neural Networks and Deep Learning, Ch. 1 and Ch. 2, without reading reference code examples
#   Gradients and backpropagation: 
#       -> 3Blue1Brown: Backpropagation, intuitively / Backpropagation calculus
#       -> ritvikmath: "Backpropogation: Data Science Concepts"
#       -> https://tutorial.math.lamar.edu/classes/calci/linearapproximations.aspx
#       -> Softmax function: https://en.wikipedia.org/wiki/Softmax_function / https://medium.com/data-science/derivative-of-the-softmax-function-and-the-categorical-cross-entropy-loss-ffceefc081d1
#   SGD:
#       -> https://en.wikipedia.org/wiki/Stochastic_gradient_descent 
#       -> https://www.geeksforgeeks.org/deep-learning/adam-optimizer/
#       -> Jacob Murri, Math 118 Lecture Notes 
#
#   Autoencoders:
#       -> Durk Kingma, Max Welling: An Introduction to Variational Autoencoders 
#               (I had to do a lot of reading to understand what they were saying, since I've never taken Bayesian stats)
#       -> 


In [1671]:
import numpy as np

In [1672]:
# =========  1: The forward pass ===========
# 1.1 Creating the linear layers 
# A linear layer's weight matrix has size (output, input)
# 
# Per-neuron outputs per scalar j are ∑_i (w_j)_i * x_i + b_j, where i is the input dimension, and j is the output dimension
# Hence the j'th row of W corresponds to the vector w_j 
# All rows dot the same vector x
# -> Output of len j is the composition W @ x + b

class Layer(): 
    index = None
    output = None
    next = None
    prev = None

class LinearLayer(Layer): 
    weights: np.array #(output_size, input_size)
    bias = np.array #(output_size, )
    def __init__(self, input_size: int, output_size: int): 
        rng = np.random.default_rng()
        self.weights = rng.random((output_size, input_size)) #Initialize with random weights
        self.bias = rng.random(output_size)    
        self.grad_function = lambda x: self.weights.T
    def __call__(self, x: np.array): 
        return self.weights @ x + self.bias
    def size(self): 
        return f"In: {self.weights.shape[-1]}, out: {self.weights.shape[0]}"
    def __repr__(self):
        return(f"LinearLayer: {self.size()}")

layer = LinearLayer(input_size=8, output_size=3)
input = np.array([1,1,1,1,1,1,1,1])
layer(input)

array([4.7659948 , 4.12246132, 5.45307934])

In [1673]:
# 1.2 Non-linear functions
# In a perceptron, we take a singular output w1x1 + w2x2 + ... wnxn, same as dot product w * x, 
# We "accept" if the sum exceeds some certain number, b, otherwise we reject
# Mathematically equivalent to H(w*x + b)
# Because of the piecewise behavior, over multiple layers a small weight change that modifies 0->1 can drastically change behavior
# sigmoid: maps R -> (0,1), smooth version of H(x)

class Nonlinear(Layer):
    name = "nonlinear"
    grad = None
    def __call__(self, z: np.array): 
        return self.function(z)
    def __repr__(self):
        return f"{self.name}"

class Sigmoid(Nonlinear):
    def __init__(self): 
        self.function = lambda z: (1+np.exp(-z))**-1
        self.name = "sigmoid"
class ReLU(Nonlinear):
    def __init__(self):
        self.function = lambda z: np.maximum(0, z)
        self.name = "relu"

# 1.21 Nonlinear function derivatives
# To prepare for the backpropagation step later, define the derivatives of the sigmoid function and ReLU


# 1.21 final activation function thru softmax
# We want to turn the final sigmoided function into a probability distribution
class Softmax(Layer):
    def __init__(self): 
        self.function = lambda x: np.exp(x) / np.sum(np.exp(x))
        # Since softmax is not a vectorized function, then each output depends on all inputs,
        # making the _functionient a jacobian matrix 
        # Although the jacobian matrix contains functions of x_i, the softmax derivative only uses 
        # simple transformations of the original output, since diagonal entries are (y_1)(1-y_1) and 
        # off-diagonal entries are -y_i*y_j, where y_i = exp(x_i).
        # The (i,j)'th entry is y_i {i=j} - y_i*y_j
        # => diag(y) - yy^T
        
    def __call__(self, z):
        return self.function(z)

In [1674]:
# 1.3 Creating a neural network
# Add linear or nonlinear modules with `add`, can be any function that works on numpy arrays
class NeuralNetwork: 
    modules: list[int] = None
    input = Layer()
    parameters = []
    def add(self, module): 
        module = module
        module.index = len(self.modules)
        if self.modules:
            module.prev = self.modules[-1]
            self.modules[-1].next = module
        else:
            module.prev = self.input
        self.modules.append(module)
        #Add parameters if it's a linear layer
        if isinstance(module, LinearLayer): 
            self.parameters.append({"weights": module.weights, "biases": module.bias})
    def __init__(self): 
        self.modules = []
        self.parameters = []
    def forward(self, input):
        self.input.output = input 
        output = input
        for module in self.modules: 
            output = module(output)
            module.output = output
        return output
    def __repr__(self):
        for i, module in enumerate(self.modules):
            print(f"module {i}: {module}")

NN = NeuralNetwork()
NN.add(module=LinearLayer(input_size=10, output_size=8))
NN.add(module=Sigmoid())
NN.add(module=LinearLayer(input_size=8, output_size=6))
NN.add(module=Sigmoid())
NN.add(module=LinearLayer(input_size=6, output_size=2))
NN.add(module=Softmax())

# A randomly initiated neural network is quite wrong
NN.forward(np.array([1,1,1,1,1,1,1,1,1,1,]))

array([0.43657863, 0.56342137])

In [1675]:
# =========  2: Using gradients to update the weights in neural networks ===========
# 2.0 Loss is a function of the weights
# We initialize our neural network with random weights and biases for the linear layers. 
# If we put in an input, the output is probably wrong in most cases.   
#
#This makes it that if we can quantify how "wrong" the random-weight answer is, 
# then it is possible to find some other setting for all the weights and biases 
# inside the neural network that make the answer *less wrong* 
#
# In this case, a neural network is a function f(weights + biases) from the 
# parameter dimension to the output dimension, therefore the loss is a
# function Loss(weights + biases) -> R
# 
# Furthermore, the neural network itself should be a differentiable function by
# the chain rule from calc, so as long as our loss function is differentiable 
# we can find the partial derivative of Loss(w + b) with respect to every individual w, b
#
# By the first order linear approximation, f(a+∆x) ≈ f(a) + f'(a)∆x 
# Therefore if we move in direction of -f'(a) we reduce f (basic principle of GD)

# 2.1 MSE 
# Since our goal is a autoencoder, which does a recreation, MSE is a more natural metric than Cross-entropy loss
#
# When a neural network with randomized weights spits out an output distribution,
# we have to score the output distribution based on something. 
# MSE is a reliable metric, the squared term punishes large errors while allowing small ones to exist. 
# This makes our loss error prioritize correcting the biggest differences from training reference while
# caring less about being perfectly precise on nearly-good weight differences
# It's nice and differentiable with the derivative being a linear function


class LossFunction(): 
    function = None
    grad = None
    def __call__(self, x, ref):
        return self.function(x, ref)

class MeanSquaredError(LossFunction):
    def __init__(self): 
        self.function = lambda x, ref: np.sum((x-ref)**2)/(2*len(x))



# 2.2 Recursive chain rule applications make it easy to find individual derivatives
# I had to write this in a markdown below for the math
# For a given output neuron, its activation depends on the weights connecting to it and 
# its bias, but also the activations that all the neurons prior to it connected 
# The later a given operation is, we can relate it to dC more easily,
# If we can anchor somewhere and find a recursive formula between layers given our repeated 
# LinearLayer + sigmoid structure, we can easily recreate the chain rule with less computation
# For instance, if we can find a relation between dCdz2, and dz2dz1, we can find dC with respect 
# To any linear layer output z. 
# 
# since z is a composition of w and b, then one more chain rule application can easily find dCdw or dCdb

$$C = \frac{1}{2n}\sum_k (\hat{y}_k - y_k)^2$$
$$\text{where the neural network output } \hat y = \text{softmax}(z^{(L)}) $$
$$ \frac{\partial C}{\partial z^{(L)}_j} = \sum_i \frac{\partial \hat y_i }{\partial z^{(L)}_j} \frac{\partial C}{\partial \hat y _i}$$
$$\sigma \text{ is a nice vectorized function, but all values } y_i \text{ depend on } z^{(L)}_j \text{ in softmax}$$
$$\frac{\partial \hat y_i}{\partial z_j} = \begin{cases} \hat y_i - \hat y_i \hat y_j, \quad i = j \\ - \hat y_i \hat y_j, \quad i \neq j \end{cases} \implies \frac{\partial \hat y}{\partial z} = \text{diag}( \hat y) - \hat y \hat y^\top \text{ (A jacobian)}$$
$$\boxed{ \frac{\partial C}{\partial z^{(L)}} = \frac{1}{n} \left((\text{diag}(\hat y) - \hat y \hat y^\top) (\hat y - y) \right)}$$

$$\frac{\partial C}{\partial z^{(L-1)}} = \frac{\partial x^{(L-1)}}{\partial z^{(L-1)}} \frac{\partial z^{(L)}}{\partial x^{(L-1)}} \frac{\partial C}{\partial z^{(L)}}$$
$$\text{break to calc derivatives...}$$

$$ x^{(L)}_i = \sigma(z^{(L)}_i) \rightarrow \frac{\partial x_i}{\partial z^{(L)}_j} = \begin{cases} \sigma(z^{(L)}_i)(1-\sigma(z^{(L)}_i)), \quad i = j \\ 0, \quad \quad i \neq j\end{cases}
$$
$$z^{(L)} = W^{(L)}x^{(L-1)} + b^{(L)} \implies \frac{\partial{z^{(L)}}}{\partial x^{(L-1)}} = (W^{(L)})^\top$$
$$ \boxed{\implies \frac{\partial C}{\partial z^{(L-1)}} = \text{diag}\left(x^{(L-1)} \odot (1-x^{(L-1)}) \right) (W^{(L)})^\top \frac{\partial C}{\partial z^{(L)}}}$$



$$\text{Recursion on weights:}$$
$$\frac{\partial C}{\partial w_{ij}} = \sum_k \frac{\partial z^{(L)}_k}{\partial w^{{(L)}}_{i,j}} \frac{\partial C}{\partial z^{(L)}_k}$$
$$z^{(L)}_k = \sum_j w_{k,j}x_j + b^{(L)}_k \implies \frac{\partial z^{(L)}_k}{\partial w^{(L)}_{i, j}} = \begin{cases} x_j^{(L)}, \quad i = k \\ 0,\quad \text{ otherwise}\end{cases}$$
$$\text{therefore } \frac{\partial C}{\partial w^{(L)}_{ij}} = \frac{\partial z_i^{(L)}}{\partial w_{i, j}^{(L)}} \frac{\partial C}{\partial z^{{(L)}}_i} = x_j \frac{\partial C}{\partial z^{(L)}_i}, \quad \left[\frac{\partial C}{\partial W^{(L)}} \right]_{ij} = \frac{\partial C}{\partial z^{(L)}_i} x^{(L-1)}_j$$
$$ \boxed{\implies \frac{\partial C}{\partial W^{(L)}} = \frac{\partial C}{\partial z^{(L)}} \otimes x^{(L-1)}}$$
$$\text{Recursion on biases:}$$
$$z^{(L)}_k = \sum_j w_{k,j}^{(L)}x_j^{(L-1)} + b^{(L)}_k \implies \frac{\partial z^{(L)}_k}{\partial b^{(L)}_{i}} = \begin{cases}1, \quad i = k \\ 0, \quad i \neq k \end{cases} $$
$$\implies \frac{\partial C}{\partial b^{(L)}_i} = \frac{\partial C}{\partial z^{(L)}_i}$$
$$\boxed{\frac{\partial C}{\partial b^{(L)}} = \frac{\partial C}{\partial z^{(L)}}}$$



In [1676]:
# Starting at the last layer, compute the first gradients dC/dz^L using only the last layer
# Then apply recursively

class loss:
    output_loss = 1
    outputs = None
    targets = None
    dCdzs = []
    dCdWs = []
    def __init__(self, model, loss_fn, outputs, targets): 
        self.NN = model
        self.loss_fn = loss_fn
        self.outputs = outputs
        self.targets = targets
        self.output_loss = loss_fn(outputs, targets)
        self.dCdzs = []
        self.dCdWs = []
    def update(self, outputs, targets):
        self.output_loss = self.loss_fn(outputs, targets) 
        return self.output_loss
    def backward(self): 
        # Uses original loss from initialization to recursively compute error with respect to layers
        # Final nonlinear output z^L  ...
        # No idea how pytorch implements this, so this may be very inefficient
        # Loss functions propagate gradients straight backwards (since they are numpy-vectorized)
        # Linear layers distribute gradients from earlier layers to affect later layers 
        # 
        # While we could technically chain rule derivatives backwards, all our neural networks
        # are setup in a (LinearLayer(x^{L-1}) -> z^{L} -> o(z) -> x ) fashion
        # Hypothetically though, we could easily calculate derivatives by defining module.grad and 
        # chaining together module.grad(module.prev.output) / module.grad(module.output) for implicit cases
        # 
        # For the final group, find dCdzL
        # For the lower groups, dCdzL_ with the recursion
        self.dCdzs.append((1/len(self.outputs)) * (np.diag(self.outputs) - np.outer(self.outputs, self.outputs)) @ (self.outputs - self.targets))
        #Walk back. When you've encountered the output of a sigmoid layer o(z) = x, find dx_dz_ and relate it to dCdz
        current_module = self.NN.modules[-1]
        xs = []
        while current_module.prev is not None: 
            current_module = current_module.prev
            if isinstance(current_module, Sigmoid): 
                x = current_module.output
                xs.append(x)
                W = current_module.next.weights #dzdx_
                a = np.diag(x * (1-x)) #dx_dz_
                dCdz_ = a @ W.T @ self.dCdzs[-1] #Chain rule with dCdz
                self.dCdzs.append(dCdz_)
        xs.append(self.NN.input.output) #x^{0} is an input to a linear layer

        #Find dCdW and update weights
        self.dCdWs = [
            np.outer(dCdz, x) for dCdz, x in zip(self.dCdzs, xs)
        ]
        #dCdb_i is just dCdz_i because of the additive derivative
        # Organize into list[dict] notation similar to model.parameters
        return [
            {"d_weights": dCdW, "d_biases": dCdb} 
            for dCdW, dCdb in zip(reversed(self.dCdWs), reversed(self.dCdzs))
        ]

class Optimizer:
    NN: NeuralNetwork
    lr: int
    def __init__(self, NN, lr): 
        self.NN = NN
        self.lr = lr

class SGD(Optimizer): 
    def __init__(self, NN, lr, momentum: float=0): 
        super().__init__(NN, lr)
        self.momentum = momentum
        self.m_prev = []
    def step(self, grad): #Stock pytorch probably puts grad into the model.parameters object, but I don't want to do that
        linear_layers = []
        for module in self.NN.modules: 
            if isinstance(module, LinearLayer):
                linear_layers.append(module)
        i = 0
        for L in linear_layers: 
            grad_W = grad[i]["d_weights"]
            grad_b = grad[i]["d_biases"]
            # on initialization, there is no m_prev, so replace m_ with zeroes
            m_W_prev = self.m_prev[i]["m_weights"] if i < len(self.m_prev) else np.zeros_like(grad_W)
            m_b_prev = self.m_prev[i]["m_biases"] if i < len(self.m_prev) else np.zeros_like(grad_b)
            # Step
            m_W = self.momentum*m_W_prev + (1-self.momentum)*grad_W
            m_b = self.momentum*m_b_prev + (1-self.momentum)*grad_b
            L.weights -= self.lr*m_W
            L.bias -= self.lr*m_b
            i += 1
            if self.momentum: #Save to same index for momentum 4 next training example
                if i < len(self.m_prev): 
                    self.m_prev[i] = {"m_weights": m_W, "m_biases": m_b} #Override
                else: 
                    self.m_prev.append({"m_weights": m_W, "m_biases": m_b}) #if it doesn't exist, just append it
            
class Adam(Optimizer):
    # With SGD, we are only approximating the first-order gradient, often badly
    # So what use would approximating the parameters^2 terms for the second derivative? 
    # Our sample gradients are from a distribution, and so if we can find the variance
    # I sort of couldn't find an explanation online for how the gradient variance works, 
    # But based on the formula, a basic idea is that:  
    #   Var(dC) = E(dC^2) - E(dC)^2 by basic stats 
    #   sqrt(E(dC^2)) ∝ E(dC) so a large gradient implies a large variance
    #   Large gradient means big steps are dangerous, so scale the learning rate proportionally by sqrt(E(dC^2))
    def __init__(self, NN, lr, b1: float, b2: float): 
        super().__init__(NN, lr)
        self.b1 = b1
        self.b2 = b2
        self.m_prev = []
        self.v_prev = []
    def step(self, grad): 
        linear_layers = []
        for module in NN.modules: 
            if isinstance(module, LinearLayer):
                linear_layers.append(module)
        i = 0
        for L in linear_layers:
            #First moment
            grad_W = grad[i]["d_weights"]
            grad_b = grad[i]["d_biases"]
            m_W_prev = self.m_prev[i]["m_weights"] if i < len(self.m_prev) else np.zeros_like(grad_W)
            m_b_prev = self.m_prev[i]["m_biases"] if i < len(self.m_prev) else np.zeros_like(grad_b)
            v_W_prev = self.v_prev[i]["v_weights"] if i < len(self.v_prev) else np.zeros_like(grad_W)
            v_b_prev = self.v_prev[i]["v_biases"] if i < len(self.v_prev) else np.zeros_like(grad_b)
            #Step
            m_W = self.b1*m_W_prev + (1-self.b1)*grad_W
            m_b = self.b1*m_b_prev + (1-self.b1)*grad_b
            v_W = self.b2*v_W_prev + (1-self.b2)*(grad_W**2)
            v_b = self.b2*v_b_prev + (1-self.b2)*(grad_b**2)

            #Actually, this is wrong, the exponent is the current training iteration, so replace it with a step counter that saves in Optimizer objects
            hat_m_W = m_W/(1-self.b1**(i+1))
            hat_m_b = m_b/(1-self.b1**(i+1))
            hat_v_W = v_W/(1-self.b2**(i+1))
            hat_v_b = v_b/(1-self.b2**(i+1))

            L.weights -= self.lr*(hat_m_W/(np.sqrt(hat_v_W)+1e-6))
            L.bias -= self.lr*(hat_m_b/(np.sqrt(hat_v_b)+1e-6))
            i += 1
            #Save at same index
            if i < len(self.m_prev): 
                self.m_prev[i] = {"m_weights": m_W, "m_biases": m_b} 
                self.v_prev[i] = {"v_weights": v_W, "v_biases": v_b} #Since they get populated at the same time, just do one length check
            else: 
                self.m_prev.append({"m_weights": m_W, "m_biases": m_b})
                self.v_prev.append({"v_weights": v_W, "v_biases": v_b})
            



In [1677]:
input = np.array([1,1,1,1,1,1,1,1,1,1,])
output = NN.forward(input)
print(f"Output: {output}")

MSE = MeanSquaredError()
l = loss(NN, MSE, output, np.array([0.7, 0.3]))
print(f"Output loss: {l.output_loss}")

grad = l.backward()

Output: [0.43657863 0.56342137]
Output loss: 0.03469540784071366


In [1678]:
s = Adam(NN, 0.01, 0.5, 0.5)
s.step(grad)

output2 = NN.forward(input)
print(f"Output: {output2}")
print(f"Output loss: {l.update(output2, np.array([0.5,0.5]))}")

Output: [0.46289602 0.53710398]
Output loss: 0.0006883527772135706


___

In [ ]:
# Variational Autoencoder
#
# Autoencoders themselves are pretty simple: minimize reconstruction error, and the model finds an approximate
# parameter distribution in its lower dimension for each category. To me, I like to think of autoencoders as a nonlinear version of PCA
# where the latent space is our low-rank projection V^Tx, with r kept components corresponding to the dimension r of the smallest network layer
# While it can find distributions here
# pretty well, an arbitrary combination in parameter space is not guaranteed to look like anything after reconstruction, 
# so it's not good at generalization/generation
# 
# The trick with VAEs is to minimize latent loss in addition to reconstruction loss, meaning that it forces the lower-dimensional
# distribution to resemble whatever prior we think the latent space has 
# The paper says how any sufficiently commplicated function can map any arbitrary distribution to another arbitrary distribution, meaning that 
# the VAE has to learn that sufficiently complicated function that can turn a nice interpolated space (e.g the gaussian) into our output distribution
#
# At a very high level, we just penalize the approximated latent distribution q(z|x) with something like KL, 
# even though the Kingma I read discusses this in much more detail I can't really understand (ELBO and true posteriors and monte-carlo estimators, etc)
#
# To be honest, I spent a long time reading over ELBO, in which the original paper used a 


Autoencoder = NeuralNetwork()
Autoencoder.add(LinearLayer(64, 32))
Autoencoder.add(Sigmoid())
Autoencoder.add(LinearLayer(32, 16))
Autoencoder.add(Sigmoid())
Autoencoder.add(LinearLayer(16, 32))
Autoencoder.add(Sigmoid())
Autoencoder.add(LinearLayer(32, 64))
Autoencoder.add(Softmax())